In [1]:
from pathlib import Path
import pandas as pd
import json
import joblib

from sklearn.preprocessing import RobustScaler

In [2]:
# ===== RUTAS =====
PROJECT_ROOT = Path.cwd().resolve().parents[3]

NOMBRE_SPLIT = "BCCC17__split__v2"
NOMBRE_SCALED = "BCCC17__scaled__v2"

RUTA_SPLIT = PROJECT_ROOT / "02_datasets" / "processed_analisis_estadistico" / NOMBRE_SPLIT
RUTA_SALIDA = PROJECT_ROOT / "02_datasets" / "processed_analisis_estadistico" / NOMBRE_SCALED

# ===== ARCHIVOS =====
TRAIN_FILE = f"{NOMBRE_SPLIT}__train.csv"
TEST_FILE = f"{NOMBRE_SPLIT}__test.csv"

TRAIN_OUT = f"{NOMBRE_SCALED}__train.csv"
TEST_OUT = f"{NOMBRE_SCALED}__test.csv"

SCALER_FILE = "scaler.pkl"
REPORT_FILE = f"{NOMBRE_SCALED}_report.json"

# ===== CONFIG =====
LABEL_COL = "LABEL"
SCALER = RobustScaler()

In [3]:
print("Ruta split:", RUTA_SPLIT)
print("Ruta salida:", RUTA_SALIDA)

Ruta split: /LUSTRE/home/inginf/u32902122/TFG/02_datasets/processed_analisis_estadistico/BCCC17__split__v2
Ruta salida: /LUSTRE/home/inginf/u32902122/TFG/02_datasets/processed_analisis_estadistico/BCCC17__scaled__v2


In [4]:
train_path = RUTA_SPLIT / TRAIN_FILE
test_path = RUTA_SPLIT / TEST_FILE

if not train_path.exists():
    raise FileNotFoundError(train_path)

if not test_path.exists():
    raise FileNotFoundError(test_path)

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (1890959, 64)
Test shape: (472740, 64)


In [5]:
X_train = train_df.drop(columns=[LABEL_COL])
y_train = train_df[LABEL_COL]

X_test = test_df.drop(columns=[LABEL_COL])
y_test = test_df[LABEL_COL]

print("Features:", X_train.shape[1])

Features: 63


In [6]:
numeric_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Columnas numéricas:", len(numeric_cols))

Columnas numéricas: 63


In [7]:
SCALER.fit(X_train[numeric_cols])

,"with_centering with_centering: bool, default=TrueIf `True`, center the data before scaling.This will cause :meth:`transform` to raise an exception when attemptedon sparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_scaling with_scaling: bool, default=TrueIf `True`, scale the data to interquartile range.",True
,"quantile_range quantile_range: tuple (q_min, q_max), 0.0 < q_min < q_max < 100.0, default=(25.0, 75.0)Quantile range used to calculate `scale_`. By default this is equal tothe IQR, i.e., `q_min` is the first quantile and `q_max` is the thirdquantile... versionadded:: 0.18","(25.0, ...)"
,"copy copy: bool, default=TrueIf `False`, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"unit_variance unit_variance: bool, default=FalseIf `True`, scale data so that normally distributed features have avariance of 1. In general, if the difference between the x-values of`q_max` and `q_min` for a standard normal distribution is greaterthan 1, the dataset will be scaled down. If less than 1, the datasetwill be scaled up... versionadded:: 0.24",False


In [8]:
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = SCALER.transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = SCALER.transform(X_test[numeric_cols])

In [9]:
train_scaled = X_train_scaled.copy()
train_scaled[LABEL_COL] = y_train

test_scaled = X_test_scaled.copy()
test_scaled[LABEL_COL] = y_test

In [10]:
RUTA_SALIDA.mkdir(parents=True, exist_ok=True)

train_scaled.to_csv(RUTA_SALIDA / TRAIN_OUT, index=False)
test_scaled.to_csv(RUTA_SALIDA / TEST_OUT, index=False)

print("Train guardado:", TRAIN_OUT)
print("Test guardado:", TEST_OUT)

Train guardado: BCCC17__scaled__v2__train.csv
Test guardado: BCCC17__scaled__v2__test.csv


In [11]:
joblib.dump(SCALER, RUTA_SALIDA / SCALER_FILE)

print("Scaler guardado en:", SCALER_FILE)

Scaler guardado en: scaler.pkl


In [12]:
reporte = {
    "dataset_entrada": NOMBRE_SPLIT,
    "scaler": "RobustScaler",
    "num_features": len(numeric_cols),
    "features_scaled": numeric_cols,
    "train_shape": train_scaled.shape,
    "test_shape": test_scaled.shape
}

with open(RUTA_SALIDA / REPORT_FILE, "w") as f:
    json.dump(reporte, f, indent=2)

print("Reporte guardado")

Reporte guardado


In [13]:
print("========== RESUMEN ==========")
print("Scaler:", type(SCALER).__name__)
print("Features escaladas:", len(numeric_cols))
print("Train:", train_scaled.shape)
print("Test:", test_scaled.shape)
print("=============================")

========== RESUMEN ==========
Scaler: RobustScaler
Features escaladas: 63
Train: (1890959, 64)
Test: (472740, 64)
